## **Table of Contents**

1. **Introduction**
    - Overview of Earth2Studio and Diffusion Models
    - Objective of the Notebook
2. **Setup and Environment**
    - Verifying the Earth2Studio Docker Image
    - Running the Earth2Studio Container
    - Installing Required Dependencies
3. **Loading the Diffusion Model**
    - Overview of the Diffusion Endpoint
    - Loading the Pretrained Diffusion Model
4. **Preparing the Input Data**
    - Understanding the Input Data Format
    - Loading and Preprocessing the Data
5. **Making Predictions**
    - Using the Diffusion Endpoint for Inference
    - Visualizing the Predictions
6. **Saving and Exporting Results**
    - Saving Predictions to Disk
    - Exporting Results for Further Analysis
7. **Conclusion**
    - Summary of Steps
    - Next Steps and Resources

---


### **3. Loading the Diffusion Model**

- **Theory**: Explain the diffusion endpoint and how it is used for inference. Mention that the model checkpoint from the training step will be loaded.
- **Practice**:
    - Load the diffusion model using Earth2Studio:
        
        from earth2studio.models import DiffusionModel
        
        model = DiffusionModel.load_from_checkpoint("/app/checkpoints_regression/UNet.0.180000.mdlus")
        
        model.eval()
        

---

### **4. Preparing the Input Data**

- **Theory**: Describe the expected input format for the diffusion model (e.g., variables, dimensions, etc.).
- **Practice**:
    - Load and preprocess the input data:
        
        import xarray as xr
        
        input_data = xr.open_dataset("/app/data/custom_data_1/ERA5_wrf_combined/custom_concat_train.nc")
        
        preprocessed_data = model.preprocess(input_data)
        

---

### **5. Making Predictions**

- **Theory**: Explain how the diffusion endpoint generates predictions and the importance of batch processing for large datasets.
- **Practice**:
    - Run inference:
        
        predictions = model.predict(preprocessed_data)
        
    - Visualize the predictions:
        
        import matplotlib.pyplot as plt
        
        plt.imshow(predictions[0])
        
        plt.title("Prediction Example")
        
        plt.show()
        

---

### **6. Saving and Exporting Results**

- **Theory**: Discuss the importance of saving predictions for further analysis.
- **Practice**:
    - Save predictions to disk:
        
        predictions.to_netcdf("/app/data/custom_data_1/predictions.nc")
        
    - Export results for external tools:
        
        predictions.to_dataframe().to_csv("/app/data/custom_data_1/predictions.csv")
        

---

### **7. Conclusion**

- **Theory**: Summarize the steps taken and the results achieved.
- **Practice**: Add a markdown cell with a summary and links to further resources (e.g., Earth2Studio documentation, diffusion model papers).

# **1. Introduction**

## Overview of Earth2Studio and Diffusion Models

Earth2Studio is a powerful framework designed for geospatial and scientific machine learning workflows. It provides tools for training, evaluating, and deploying machine learning models, including advanced diffusion models. Diffusion models are a class of generative models that learn to generate data by iteratively refining noise into meaningful patterns. These models are particularly effective for tasks like image synthesis, data imputation, and scientific simulations.

In this notebook, we will use Earth2Studio's diffusion endpoint to perform predictions on custom data. The diffusion model was trained in the previous notebook (`2-training.ipynb`) on a regression task using your custom dataset. Now, we will leverage the trained model to make predictions and analyze the results.

---

## Objective of the Notebook

The primary objective of this notebook is to:
1. Load the pretrained diffusion model from Earth2Studio.
2. Prepare the input data for inference.
3. Use the diffusion endpoint to make predictions.
4. Visualize and save the results for further analysis.

By the end of this notebook, you will have a clear understanding of how to use Earth2Studio's diffusion endpoint for inference on custom data.

# **2. Setup and Environment**

## Verifying the Earth2Studio Docker Image

Before running the Earth2Studio container, ensure that the Docker image has been built successfully. You can verify this by listing the available Docker images and checking for the `earth2studio` image:

    docker images | grep earth2studio

You should see an entry similar to this:

    REPOSITORY           TAG       IMAGE ID       CREATED          SIZE
    earth2studio         latest    <IMAGE_ID>     <CREATED_DATE>   <SIZE>

---

## Running the Earth2Studio Container

To run the Earth2Studio container in interactive mode with GPU support, use the following command:

    bash scripts/run_docker_e2s.sh

This script mounts the necessary directories (e.g., `data`, `scripts`, `notebooks`, `outputs`) into the container and starts it with GPU support. Once the container is running, you will have access to the Earth2Studio environment.

---

## Installing Required Dependencies

The Earth2Studio environment has already been set up in the Docker image. The following steps were performed during the image creation:

1. Installed system dependencies such as `git`, `make`, `curl`, `python3.11`, and `pip`.
2. Installed Earth2Studio and its required Python packages using the following commands:

        uv pip install --system --break-system-packages ".[dlwp,corrdiff]"
        pip install -r requirements_docker.txt

3. Set up the environment to avoid permission issues with Jupyter Notebook.

If you need to verify the installation inside the container, you can run the following commands:

    # Check Python version
    python3 --version

    # Check installed Python packages
    pip list

At this point, your environment is ready to use Earth2Studio for running diffusion models.

# **3. Loading the Diffusion Model**

## Overview of the Diffusion Endpoint

The diffusion endpoint in Earth2Studio is designed to load and use pretrained diffusion models for inference. These models are trained to iteratively refine noisy data into meaningful patterns, making them highly effective for tasks such as image synthesis, data imputation, and scientific simulations. In this section, we will load the pretrained diffusion model checkpoint that was created during the training phase.

---

## Loading the Pretrained Diffusion Model

To load the pretrained diffusion model, we will use the `DiffusionModel` class provided by Earth2Studio. The model checkpoint from the training step will be loaded, and the model will be set to evaluation mode.

### Code Example:

In [12]:
! ls /app/outputs/checkpoints_diffusion

EDMPrecondSuperResolution.0.1000.mdlus	 checkpoint.0.1000.pt
EDMPrecondSuperResolution.0.10000.mdlus  checkpoint.0.10000.pt
EDMPrecondSuperResolution.0.10504.mdlus  checkpoint.0.10504.pt
EDMPrecondSuperResolution.0.11000.mdlus  checkpoint.0.11000.pt
EDMPrecondSuperResolution.0.11504.mdlus  checkpoint.0.11504.pt
EDMPrecondSuperResolution.0.12000.mdlus  checkpoint.0.12000.pt
EDMPrecondSuperResolution.0.12504.mdlus  checkpoint.0.12504.pt
EDMPrecondSuperResolution.0.13000.mdlus  checkpoint.0.13000.pt
EDMPrecondSuperResolution.0.13504.mdlus  checkpoint.0.13504.pt
EDMPrecondSuperResolution.0.14000.mdlus  checkpoint.0.14000.pt
EDMPrecondSuperResolution.0.14504.mdlus  checkpoint.0.14504.pt
EDMPrecondSuperResolution.0.15000.mdlus  checkpoint.0.15000.pt
EDMPrecondSuperResolution.0.1504.mdlus	 checkpoint.0.1504.pt
EDMPrecondSuperResolution.0.15504.mdlus  checkpoint.0.15504.pt
EDMPrecondSuperResolution.0.16000.mdlus  checkpoint.0.16000.pt
EDMPrecondSuperResolution.0.16504.mdlus  checkpoint.0.16504

In [16]:
import torch

checkpoint_path = "/app/outputs/checkpoints_diffusion/checkpoint.0.9504.pt"
checkpoint = torch.load(checkpoint_path, map_location="cuda")
print(checkpoint.keys())  # List the keys in the checkpoint

/tmp/ipykernel_59/1509339341.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cuda")


dict_keys(['optimizer_state_dict', 'epoch'])


In [18]:
import torch

mdl_path = "/app/outputs/checkpoints_diffusion/EDMPrecondSuperResolution.0.9504.mdlus"

try:
    mdl_checkpoint = torch.load(mdl_path, map_location="cpu")
    print(mdl_checkpoint.keys())  # List the keys in the checkpoint
except Exception as e:
    print(f"Error loading checkpoint: {e}")

Error loading checkpoint: "filename 'storages' not found"


/tmp/ipykernel_59/770617815.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  mdl_checkpoint = torch.load(mdl_path, map_location="cpu")


In [14]:
import torch
from earth2studio.models.dx.corrdiff import CorrDiffTaiwan

# Load the checkpoint
checkpoint_path = "/app/outputs/checkpoints_diffusion/EDMPrecondSuperResolution.0.9504.mdlus"
checkpoint = torch.load(checkpoint_path, map_location="cuda")  # or "cpu"

# Initialize the model
model = CorrDiffTaiwan(**checkpoint["model_args"])  # Pass model arguments from the checkpoint
model.load_state_dict(checkpoint["state_dict"])  # Load model weights

# Set the model to evaluation mode
model.eval()

/tmp/ipykernel_59/1467367073.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cuda")  # or "cpu"


KeyError: "filename 'storages' not found"

In [1]:
from earth2studio.models import DiffusionModel

ImportError: cannot import name 'DiffusionModel' from 'earth2studio.models' (unknown location)

In [ ]:
from earth2studio.models import DiffusionModel

# Load the pretrained diffusion model checkpoint
model = DiffusionModel.load_from_checkpoint("/app/checkpoints_regression/UNet.0.180000.mdlus")

# Set the model to evaluation mode
model.eval()


This code snippet initializes the diffusion model using the specified checkpoint file and prepares it for inference. Ensure that the checkpoint file path is correct and accessible within the container.